# Rapport – Filmrekommendationssystem

1. Metod

I detta projekt implementeras ett rekommendationssystem baserat på content-based filtering. Systemet rekommenderar filmer utifrån likhet i innehåll, baserat på genres och användargenererade tags. För att representera filmernas innehåll numeriskt används TF-IDF-vektorisering av genrer och tags, och likhet mellan filmer beräknas sedan med cosine similarity varpå de filtreras efter år och rating. Cosine similarity används eftersom den mäter vinkeln mellan vektorer snarare än deras längd, vilket gör metoden robust mot skillnader i textlängd och särskilt lämplig för TF-IDF-representationer.  

Collaborative filtering övervägdes, men det kräver en mycket stor användar-film-matris vilket hade varit minneskrävande (~26 GB) och därför valdes en content-based metod istället.  

2. Datahantering

Datasetet som används är hämtat från MovieLens och består av:  

movies.csv (filmer och genres)  
ratings.csv (betyg)  
tags.csv (nyckelord)  
links.csv (kopplingar till andra filmdatabaser)  

Genres är relativt breda kategorier och kompletteras därför med tags, som innehåller mer detaljerad information för bättre träffsäkerhet. Jag aggregerade först tags på movieId för att samla alla tags per film och när jag sedan lade till tags i dataframen movies ändrade jag alla NaNs till tomma strängar för att de inte skulle påverka vid cosine similarity.  

Då ratings var den överlägset största filen med nästan 34 miljoner rader valde jag att ta genomsnittet av varje betyg per film, varpå jag lade till en rating-kolumn och en year-kolumn i movies.  

3. Arbetsprocess

Arbetet inleddes med en explorativ analys av datasetet i en Jupyter Notebook (labb1.ipynb), där strukturen i movies, ratings tags och links undersöktes. I detta steg testades även enklare rekommendationsmetoder baserade enbart på genres.

Utifrån dessa experiment utvecklades en första version av rekommendationsfunktionen (labb1.py), som därefter iterativt förbättrades genom att inkludera tags samt TF-IDF-vektorisering. Under denna process utvärderades olika designval, exempelvis viktning mellan genres och tags samt antal input-filmer.

När en stabil modell uppnåtts separerades logiken i modulära funktioner (recommender.py), vilket möjliggjorde integration i en Dash-applikation (app.py). Denna uppdelning gör systemet mer strukturerat och återanvändbart.
  
4. Feature Engineering

Genres och tags kombineras till en gemensam representation men vektoriseras separat med TF-IDF för att kunna ge tags mer vikt, detta testade jag mig fram till då jag först tänkt att tags skulle ha alldeles för mycket orelevant information. Jag upptäckte dock snabbt att rekommendationerna var mycket bättre när jag gav tags mer vikt än genrer, dock inte för mycket mer för då spelade genrer inte längre någon roll. Jag landade därför på att tags väger 1,5 gånger mer än genrer.  

TF-IDF används för att vikta ord baserat på hur unika de är för en film, vilket gör att vanliga ord får mindre betydelse medan mer specifika tags får större påverkan. Till skillnad från one-hot encoding fångar TF-IDF alltså även hur viktiga olika ord är, vilket ger en mer nyanserad representation av filmers innehåll.  
  
5. Rekommendationsalgoritm

För en vald film beräknas likheten mot alla andra filmer, likheten omvandlas till en ranking och här valde jag även att lägga till möjligheten att ange upp till tre filmer att basera rekommendationerna på för att ytterligare öka träffsäkerheten och vid flera valda filmer kombineras då deras rankingar. Genom att omvandla likhetsvärden till rankingar och summera dessa kan filmer som är konsekvent lika flera valda filmer prioriteras, vilket ger mer stabila rekommendationer. Antalet filmer att basera rekommendationerna på experimenterade jag med och jag landade på max tre filmer för att om jag gick över det så föll hela systemet och rekommendationerna blev nästan uteslutande väldigt obskyra titlar, detta attribuerade jag till att det blev för mycket brus med såpass många tags.   

De 50 mest relevanta filmerna väljs sedan ut och här säkerställs även att ens ett, två eller tre val av filmer inte är bland dem varpå rekommendationerna filtreras baserat på år, ±10 år från vald film eller årsspannet av valda filmer för att säkerställa att rekommendationerna är från ungefär samma tid.

Slutligen väljs de fem rekommendationerna baserat på vald sortering där man får välja ifall man vill ha antingen de med högst rating, de med närmast rating till ens val eller, om man är en riktig sunkfilmskonnässör, de med lägst rating. Varje rekommendation är även en länk till dess sida på IMDb vilket jag gjorde genom att matcha dess movieId med dess imdbId i links-filen och med det har jag då använt alla de fyra ovan nämnda delarna av datasetet.  
  
6. Implementation

Systemet är implementerat i Python med:

pandas (datahantering)  
scikit-learn (TF-IDF och cosine similarity)  
dash (användargränssnitt)  

Användaren kan:

välja 1–3 filmer  
få fem rekommendationer  
välja vad för rating rekommendationerna ska ha  
gå direkt till IMDb-sidan för varje rekommendation  
  
7. Förbättringar

Jämfört med en enkel modell har följande förbättringar implementerats:

kombination av genres och tags  
TF-IDF istället för enkel encoding  
viktning av features  
stöd för flera input-filmer  
filtrering baserat på år  

Möjliga förbättringar på modellen inkluderar men är inte begränsat till:  

implementering av collaborative filtering utöver content-based filtering   
sortering/uppstädning bland tags för att bättre kunna fånga exempelvis skådespelares eller regissörers namn, studio, ursprungsland m.m.  
filtrering av hollywood-filmer och ej hollywood-filmer för att undvika obskyra titlar om det inte är vad man letar efter  
  
8. Begränsningar

Begränsningar för modellen inkluderar men är inte begränsat till:  

ingen användardata används (ej collaborative filtering)  
TF-IDF fångar inte semantiska samband fullt ut  
tags kan vara inkonsekventa  
resultaten påverkas av hur data är fördelad  
  
9. Slutsats

Systemet kan med relativt hög träffsäkerhet generera relevanta rekommendationer baserat på innehållslikhet mellan filmer, men det finns fortfarande många möjliga förbättringar för att rekommendationerna ska vara bättre överlag samt för att öka användarvänligheten.  